# STEA on TKGL-Wikidata Cross-Lingual (all pairs)

PyTorch Colab notebook for **STEA** (Cai et al., COLING 2022) on **tkgl_wikidata_crosslingual**.

### Zip layout
```
tkgl_wikidata_crosslingual.zip
  en_fa/   en_zh/   en_ja/   en_fr/
    alignment_all.tsv
    entity2id_en.tsv
    entity2id_<lang>.tsv
    quadruples_en.tsv
    quadruples_<lang>.tsv
    relation2id.tsv
```

Each pair uses **4-column** quadruples: `head_id, rel_id, tail_id, ts` (integer time-step IDs).

### Scale (example: en_fa)
| | EN | other |
|---|-----|-----|
| Entities | up to ~1.2M | tens of thousands |
| Quadruples | millions | millions |
| Alignments | tens of thousands | |

### Memory design
- Sparse adjacency over entities in quadruples
- Per-entity time dictionaries
- **Batched** embedding similarity + **on-the-fly** temporal Jaccard (no dense N1 x N2 matrix)

### Colab setup
1. Runtime → **GPU** (High-RAM recommended)
2. Upload `tkgl_wikidata_crosslingual.zip` to Drive or `/content`
3. Set `LANG_PAIR` (`en_fa` / `en_zh` / `en_ja` / `en_fr`)
4. Run all cells

In [1]:
# ============================================================
# 1. Imports and reproducibility
# ============================================================

import os
import gc
import zipfile
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("WARNING: enable a Colab GPU runtime.")

PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4
GPU memory (GB): 15.64


In [2]:
# ============================================================
# 2. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ============================================================
# 3. Configuration
# ============================================================

# Choose one pair inside the zip
LANG_PAIR = "en_fr"  # en_fa | en_zh | en_ja | en_fr

# Path to the full multi-pair zip (or a single-pair en_fa.zip fallback)
ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/tkgl_wikidata_crosslingual.zip"),
    Path("/content/tkgl_wikidata_crosslingual.zip"),
    Path("/content/drive/MyDrive/en_fr.zip"),  # single-pair fallback
    Path("/content/en_fr.zip"),
]

EXTRACT_ROOT = Path("/content/tkgl_wikidata_crosslingual")

# STEA hyperparameters (paper defaults)
TRAIN_SEED = 200
DIM = 100
DEPTH = 2
MARGIN = 3.0
DROPOUT = 0.3
ALPHA = 0.3
LR = 0.005
EPOCHS_PER_ITER = 1200
ITERATIONS = 5
USE_ITERATIVE_STEA = True

CSLS_K = 10
EVAL_BATCH = 256

# Optional subsample for debugging (None = all)
MAX_QUADS_EN = None
MAX_QUADS_OTHER = None

print("LANG_PAIR:", LANG_PAIR)
print("DEVICE:", DEVICE)
print("TRAIN_SEED:", TRAIN_SEED)

LANG_PAIR: en_fr
DEVICE: cuda
TRAIN_SEED: 200


In [4]:
# ============================================================
# 4. Locate zip, extract, resolve pair folder
# ============================================================

zip_path = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if zip_path is None:
    found = list(Path("/content").rglob("*wikidata*crosslingual*.zip"))
    found += list(Path("/content").rglob("en_fa.zip"))
    zip_path = found[0] if found else None
if zip_path is None:
    raise FileNotFoundError(
        "tkgl_wikidata_crosslingual.zip (or en_fa.zip) not found. "
        "Upload to Drive or /content."
    )

print("Using zip:", zip_path)

# Extract once
marker = EXTRACT_ROOT / ".extracted"
if not marker.exists():
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print("Extracting (may take several minutes for the full zip)...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(EXTRACT_ROOT)
    marker.write_text("ok")

# Find LANG_PAIR folder that contains alignment_all.tsv
candidates = []
for p in EXTRACT_ROOT.rglob("alignment_all.tsv"):
    parent = p.parent
    if parent.name == LANG_PAIR or LANG_PAIR in str(parent):
        candidates.append(parent)

if not candidates:
    # list available pairs
    avail = sorted({p.parent.name for p in EXTRACT_ROOT.rglob("alignment_all.tsv")})
    raise FileNotFoundError(
        f"Pair '{LANG_PAIR}' not found. Available folders with alignment_all.tsv: {avail}"
    )

PAIR_DIR = candidates[0]
print("PAIR_DIR:", PAIR_DIR)
for p in sorted(PAIR_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:24s} {p.stat().st_size/1e6:.1f} MB")

# Detect language suffix from folder name (en_fa -> fa)
parts = LANG_PAIR.split("_")
if len(parts) != 2:
    raise ValueError(f"LANG_PAIR should look like en_fa, got {LANG_PAIR}")
lang_a, lang_b = parts[0], parts[1]
print(f"Languages: {lang_a} <-> {lang_b}")

Using zip: /content/drive/MyDrive/tkgl_wikidata_crosslingual.zip
PAIR_DIR: /content/tkgl_wikidata_crosslingual/tkgl_wikidata_crosslingual/en_fr
  alignment_all.tsv        29.6 MB
  entity2id_en.tsv         45.7 MB
  entity2id_fr.tsv         18.3 MB
  quadruples_en.tsv        230.6 MB
  quadruples_fr.tsv        132.4 MB
  relation2id.tsv          0.0 MB
Languages: en <-> fr


In [5]:
# ============================================================
# 5. Load alignment + quadruples (4-col: h, r, t, ts)
# ============================================================

def load_alignment(path):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        header = f.readline()
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                pairs.append((int(parts[0]), int(parts[1])))
    return pairs


def load_quadruples(path, max_rows=None):
    triples, entities, relations, times = [], set(), set(), set()
    with open(path, "r", encoding="utf-8") as f:
        header = f.readline()
        for i, line in enumerate(f):
            if max_rows is not None and i >= max_rows:
                break
            parts = line.split()
            if len(parts) < 4:
                continue
            h, r, t, ts = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
            triples.append((h, r, t, ts))
            entities.update((h, t))
            relations.add(r)
            times.add(ts)
    return triples, entities, relations, times


quad_a = PAIR_DIR / f"quadruples_{lang_a}.tsv"
quad_b = PAIR_DIR / f"quadruples_{lang_b}.tsv"
align_path = PAIR_DIR / "alignment_all.tsv"

for p in (quad_a, quad_b, align_path):
    if not p.exists():
        raise FileNotFoundError(p)

print("Loading alignment...")
all_pairs_raw = load_alignment(align_path)
print(f"  pairs: {len(all_pairs_raw):,}")

print(f"Loading {lang_a} quadruples...")
triples1, e1_raw, r1_raw, t1_raw = load_quadruples(quad_a, MAX_QUADS_EN)
print(f"  triples={len(triples1):,} ents={len(e1_raw):,} rels={len(r1_raw)} times={len(t1_raw)}")

print(f"Loading {lang_b} quadruples...")
triples2, e2_raw, r2_raw, t2_raw = load_quadruples(quad_b, MAX_QUADS_OTHER)
print(f"  triples={len(triples2):,} ents={len(e2_raw):,} rels={len(r2_raw)} times={len(t2_raw)}")

print("Raw entity-ID overlap:", len(e1_raw & e2_raw))

Loading alignment...
  pairs: 484,783
Loading en quadruples...
  triples=9,646,258 ents=1,196,407 rels=592 times=2025
Loading fr quadruples...
  triples=5,653,757 ents=506,719 rels=525 times=2025
Raw entity-ID overlap: 506719


In [6]:
# ============================================================
# 6. Remap IDs + seed/eval split
# ============================================================

def make_maps(id_set):
    ids = sorted(id_set)
    return {x: i for i, x in enumerate(ids)}


e1_map = make_maps(e1_raw)
e2_map = make_maps(e2_raw)
r_map = make_maps(r1_raw | r2_raw)

N1 = len(e1_map)
N2 = len(e2_map)
NUM_ENTITIES = N1 + N2
NUM_RELATIONS = len(r_map)


def remap_triples(triples, e_map, r_map, entity_offset=0):
    out = []
    for h, r, t, ts in triples:
        if h not in e_map or t not in e_map or r not in r_map:
            continue
        out.append((e_map[h] + entity_offset, r_map[r], e_map[t] + entity_offset, ts))
    return out


print("Remapping...")
triples1_m = remap_triples(triples1, e1_map, r_map, 0)
triples2_m = remap_triples(triples2, e2_map, r_map, N1)
del triples1, triples2
gc.collect()

valid_pairs = []
for a, b in all_pairs_raw:
    if a in e1_map and b in e2_map:
        valid_pairs.append((e1_map[a], e2_map[b] + N1))
valid_pairs = np.asarray(valid_pairs, dtype=np.int64)
np.random.shuffle(valid_pairs)

if len(valid_pairs) < TRAIN_SEED:
    raise ValueError(f"Only {len(valid_pairs)} valid pairs, TRAIN_SEED={TRAIN_SEED}")

seed_pairs = valid_pairs[:TRAIN_SEED].copy()
eval_pairs = valid_pairs[TRAIN_SEED:].copy()

print(f"N1 ({lang_a})={N1:,}  N2 ({lang_b})={N2:,}  total={NUM_ENTITIES:,}")
print(f"relations={NUM_RELATIONS}")
print(f"mapped triples: {lang_a}={len(triples1_m):,}  {lang_b}={len(triples2_m):,}")
print(f"valid pairs={len(valid_pairs):,}  seeds={len(seed_pairs)}  eval={len(eval_pairs):,}")

Remapping...
N1 (en)=1,196,407  N2 (fr)=506,719  total=1,703,126
relations=595
mapped triples: en=9,646,258  fr=5,653,757
valid pairs=484,783  seeds=200  eval=484,583


In [7]:
# ============================================================
# 7. Graph, relation incidence, time dictionaries
# ============================================================

print("Building neighborhoods and time dicts...")
neighbors = [set() for _ in range(NUM_ENTITIES)]
entity_relations = [set() for _ in range(NUM_ENTITIES)]
time_dict = [set() for _ in range(NUM_ENTITIES)]

for h, r, t, ts in triples1_m + triples2_m:
    neighbors[h].add(t)
    neighbors[t].add(h)
    entity_relations[h].add(r)
    entity_relations[t].add(r)
    time_dict[h].add(ts)
    time_dict[t].add(ts)

del triples1_m, triples2_m
gc.collect()

print("Building sparse adjacency...")
rows, cols = [], []
for i in range(NUM_ENTITIES):
    rows.append(i); cols.append(i)
    for j in neighbors[i]:
        rows.append(i); cols.append(j)

rows = np.asarray(rows, dtype=np.int64)
cols = np.asarray(cols, dtype=np.int64)
degree = np.bincount(rows, minlength=NUM_ENTITIES).astype(np.float32)
values = (1.0 / degree[rows]).astype(np.float32)

adj = torch.sparse_coo_tensor(
    torch.from_numpy(np.vstack([rows, cols])),
    torch.from_numpy(values),
    size=(NUM_ENTITIES, NUM_ENTITIES),
).coalesce().to(DEVICE)

del rows, cols, values
gc.collect()

er_rows, er_cols = [], []
for eid, rels in enumerate(entity_relations):
    for rid in rels:
        er_rows.append(eid)
        er_cols.append(rid)

relation_incidence = torch.sparse_coo_tensor(
    torch.tensor([er_rows, er_cols], dtype=torch.long),
    torch.ones(len(er_rows), dtype=torch.float32),
    size=(NUM_ENTITIES, NUM_RELATIONS),
).coalesce()

relation_count = torch.zeros(NUM_ENTITIES, 1, dtype=torch.float32)
for eid, rels in enumerate(entity_relations):
    relation_count[eid, 0] = max(len(rels), 1.0)

del er_rows, er_cols, neighbors, entity_relations
gc.collect()

print("adj nnz:", adj._nnz())
print("avg degree:", float(degree.mean()))
print("empty time dicts:", sum(1 for s in time_dict if not s))

Building neighborhoods and time dicts...
Building sparse adjacency...


/tmp/ipykernel_14323/2800272512.py:33: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adj = torch.sparse_coo_tensor(


adj nnz: 7231388
avg degree: 4.245995998382568
empty time dicts: 0


In [8]:
# ============================================================
# 8. STEA model + loss
# ============================================================

class STEA(nn.Module):
    def __init__(self, num_entities, num_relations, dim=100, depth=2, dropout=0.3):
        super().__init__()
        self.dim = dim
        self.depth = depth
        self.entity_emb = nn.Parameter(torch.empty(num_entities, dim))
        self.relation_emb = nn.Parameter(torch.empty(num_relations, dim))
        nn.init.xavier_uniform_(self.entity_emb)
        nn.init.xavier_uniform_(self.relation_emb)
        self.dropout = nn.Dropout(dropout)

    def forward(self, adj):
        structural = torch.sparse.mm(adj, self.entity_emb)
        relational = torch.sparse.mm(
            relation_incidence.to(self.entity_emb.device), self.relation_emb
        )
        relational = relational / relation_count.to(self.entity_emb.device).clamp_min(1.0)
        h = torch.cat([structural, relational], dim=1)
        representations = [h]
        for _ in range(self.depth - 1):
            h = torch.sparse.mm(adj, h)
            h = F.relu(h)
            h = self.dropout(h)
            representations.append(h)
        return torch.cat(representations, dim=1)


def stea_triplet_loss(emb, pairs, margin=3.0):
    if len(pairs) == 0:
        return emb.sum() * 0.0
    device = emb.device
    p = torch.as_tensor(pairs, dtype=torch.long, device=device)
    src, tgt = p[:, 0], p[:, 1]
    corrupt_source = torch.rand(len(p), device=device) < 0.5
    neg_src, neg_tgt = src.clone(), tgt.clone()
    n_src = int(corrupt_source.sum().item())
    n_tgt = int((~corrupt_source).sum().item())
    if n_src:
        neg_src[corrupt_source] = torch.randint(0, N1, (n_src,), device=device)
    if n_tgt:
        neg_tgt[~corrupt_source] = torch.randint(N1, N1 + N2, (n_tgt,), device=device)
    pos_dist = torch.abs(emb[src] - emb[tgt]).sum(dim=1)
    neg_dist = torch.abs(emb[neg_src] - emb[neg_tgt]).sum(dim=1)
    return F.relu(pos_dist - neg_dist + margin).mean()


model = STEA(NUM_ENTITIES, NUM_RELATIONS, dim=DIM, depth=DEPTH, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.RMSprop(model.parameters(), lr=LR)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Parameters: {n_params:,}  (~{n_params * 4 / 1e6:.1f} MB fp32)")

STEA(
  (dropout): Dropout(p=0.3, inplace=False)
)
Parameters: 170,372,100  (~681.5 MB fp32)


In [9]:
# ============================================================
# 9. Memory-efficient evaluation
# ============================================================

def temporal_jaccard_vec(src_id, target_local_ids, time_dicts, n1):
    s = time_dicts[int(src_id)]
    m = len(s)
    out = np.zeros(len(target_local_ids), dtype=np.float32)
    if m == 0:
        return out
    for j, tloc in enumerate(target_local_ids):
        tset = time_dicts[n1 + int(tloc)]
        n = len(tset)
        if n == 0:
            continue
        c = len(s & tset)
        if c:
            out[j] = (2.0 * c) / (m + n)
    return out


def csls_knn_scores(sim_row, k=10):
    k = min(k, sim_row.shape[-1])
    topk = np.partition(sim_row, -k, axis=1)[:, -k:]
    src_r = topk.mean(axis=1, keepdims=True)
    return 2.0 * sim_row - src_r


@torch.no_grad()
def evaluate_embeddings(emb, eval_pairs, alpha=0.3, csls_k=10, batch_size=256):
    emb = emb.detach()
    source_all = F.normalize(emb[:N1], p=2, dim=1)
    target_all = F.normalize(emb[N1:], p=2, dim=1)

    pairs = np.asarray(eval_pairs, dtype=np.int64)
    src_ids = pairs[:, 0]
    tgt_local = pairs[:, 1] - N1

    ranks_l2r = []
    all_tgt_local = np.arange(N2, dtype=np.int64)

    for start in range(0, len(src_ids), batch_size):
        end = min(start + batch_size, len(src_ids))
        batch_src = src_ids[start:end]
        batch_tgt = tgt_local[start:end]

        s_emb = source_all[batch_src]
        emb_sim = (s_emb @ target_all.T).cpu().numpy()
        emb_sim = csls_knn_scores(emb_sim, k=csls_k)

        time_rows = np.stack([
            temporal_jaccard_vec(int(s), all_tgt_local, time_dict, N1)
            for s in batch_src
        ], axis=0)

        final = (1.0 - alpha) * emb_sim + alpha * time_rows

        for i in range(end - start):
            scores = final[i]
            gold = int(batch_tgt[i])
            rank = int(np.sum(scores > scores[gold])) + 1
            ranks_l2r.append(rank)

        if (start // batch_size) % 20 == 0:
            print(f"  eval progress {end}/{len(src_ids)}")

    ranks = np.asarray(ranks_l2r, dtype=np.float64)
    metrics = {
        "Hits@1": float(np.mean(ranks <= 1)),
        "Hits@5": float(np.mean(ranks <= 5)),
        "Hits@10": float(np.mean(ranks <= 10)),
        "MRR": float(np.mean(1.0 / ranks)),
    }
    return {"left_to_right": metrics, "ranks": ranks}


def print_metrics(result, title="Evaluation"):
    print("\n" + title)
    print("-" * len(title))
    for k, v in result["left_to_right"].items():
        if k.startswith("Hits"):
            print(f"  {k:8s}: {v*100:.2f}%")
        else:
            print(f"  {k:8s}: {v:.4f}")


@torch.no_grad()
def mutual_nearest_pairs_from_emb(emb, existing_pairs, alpha=0.3, max_sources=None):
    existing = {(int(a), int(b)) for a, b in existing_pairs}
    left_cands = sorted({int(a) for a, b in valid_pairs if (int(a), int(b)) not in existing})
    if max_sources is not None:
        left_cands = left_cands[:max_sources]

    emb_n = F.normalize(emb, p=2, dim=1)
    target = emb_n[N1:]
    best_tgt = {}

    for start in range(0, len(left_cands), EVAL_BATCH):
        batch = left_cands[start:start + EVAL_BATCH]
        s = (emb_n[batch] @ target.T).cpu().numpy()
        for i, src in enumerate(batch):
            trow = temporal_jaccard_vec(src, np.arange(N2), time_dict, N1)
            scores = (1.0 - alpha) * s[i] + alpha * trow
            j = int(np.argmax(scores))
            best_tgt[src] = N1 + j

    new_pairs = []
    for src, tgt in best_tgt.items():
        t_emb = emb_n[tgt:tgt + 1]
        s_emb = emb_n[left_cands]
        sim = (s_emb @ t_emb.T).squeeze(1).cpu().numpy()
        best_src, best_score = None, -1e9
        tset = time_dict[tgt]
        for idx, s_id in enumerate(left_cands):
            sset = time_dict[s_id]
            m, n = len(sset), len(tset)
            tscore = (2.0 * len(sset & tset) / (m + n)) if (m and n) else 0.0
            score = (1.0 - alpha) * float(sim[idx]) + alpha * tscore
            if score > best_score:
                best_score = score
                best_src = s_id
        if best_src == src and (src, tgt) not in existing:
            new_pairs.append((src, tgt))

    return np.asarray(new_pairs, dtype=np.int64).reshape(-1, 2)

print("Eval helpers ready.")

Eval helpers ready.


In [ ]:
# ============================================================
# 10. Training loop
# ============================================================

def train_one_iteration(model, optimizer, train_pairs, epochs=1200):
    model.train()
    train_pairs_np = np.asarray(train_pairs, dtype=np.int64)
    history = []
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        emb = model(adj)
        perm = np.random.permutation(len(train_pairs_np))
        loss = stea_triplet_loss(emb, train_pairs_np[perm], margin=MARGIN)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        history.append(float(loss.detach().cpu()))
        if epoch == 1 or epoch % 100 == 0 or epoch == epochs:
            print(f"Epoch {epoch:4d}/{epochs} | loss={history[-1]:.6f}")
    return history


current_train_pairs = seed_pairs.copy()
n_iterations = ITERATIONS if USE_ITERATIVE_STEA else 1

for iteration in range(1, n_iterations + 1):
    print("\n" + "=" * 70)
    print(f"STEA {LANG_PAIR} | iteration {iteration}/{n_iterations} | train={len(current_train_pairs)}")
    print("=" * 70)

    train_one_iteration(model, optimizer, current_train_pairs, epochs=EPOCHS_PER_ITER)

    model.eval()
    with torch.no_grad():
        final_emb = model(adj)

    eval_subset = (
        eval_pairs if len(eval_pairs) <= 5000
        else eval_pairs[np.random.choice(len(eval_pairs), 5000, replace=False)]
    )
    print(f"Evaluating on {len(eval_subset)} pairs...")
    result = evaluate_embeddings(
        final_emb, eval_subset, alpha=ALPHA, csls_k=CSLS_K, batch_size=EVAL_BATCH
    )
    print_metrics(result, f"Eval after iteration {iteration}")

    if not USE_ITERATIVE_STEA or iteration == n_iterations:
        break

    print("Mutual-NN pseudo pairs...")
    new_pairs = mutual_nearest_pairs_from_emb(
        final_emb, current_train_pairs, alpha=ALPHA
    )
    print("New pairs:", len(new_pairs))
    if len(new_pairs) == 0:
        print("No new pairs; stop.")
        break
    current_train_pairs = np.vstack([current_train_pairs, new_pairs])

print("\nFinal training-pair count:", len(current_train_pairs))


STEA en_fr | iteration 1/5 | train=200
Epoch    1/1200 | loss=0.134219
Epoch  100/1200 | loss=0.055746
Epoch  200/1200 | loss=0.000000
Epoch  300/1200 | loss=0.002761
Epoch  400/1200 | loss=0.000000
Epoch  500/1200 | loss=0.000000
Epoch  600/1200 | loss=0.002275
Epoch  700/1200 | loss=0.000000
Epoch  800/1200 | loss=0.006824
Epoch  900/1200 | loss=0.000000
Epoch 1000/1200 | loss=0.007931
Epoch 1100/1200 | loss=0.000000
Epoch 1200/1200 | loss=0.000000
Evaluating on 5000 pairs...
  eval progress 256/5000

Eval after iteration 1
----------------------
  Hits@1  : 44.48%
  Hits@5  : 58.14%
  Hits@10 : 63.76%
  MRR     : 0.5111
Mutual-NN pseudo pairs...


In [ ]:
# ============================================================
# 11. Final evaluation on all held-out pairs
# ============================================================

model.eval()
with torch.no_grad():
    final_embeddings = model(adj)

print(f"Full eval on {len(eval_pairs):,} pairs...")
final_result = evaluate_embeddings(
    final_embeddings, eval_pairs, alpha=ALPHA, csls_k=CSLS_K, batch_size=EVAL_BATCH
)
print_metrics(final_result, f"FINAL STEA RESULT ({LANG_PAIR}, left->right)")

print("\nConfiguration")
print("-------------")
for k in ["LANG_PAIR", "TRAIN_SEED", "DIM", "DEPTH", "MARGIN", "DROPOUT",
          "ALPHA", "LR", "EPOCHS_PER_ITER", "ITERATIONS", "CSLS_K"]:
    print(f"{k}: {globals()[k]}")
print(f"N1={N1:,} N2={N2:,} NUM_RELATIONS={NUM_RELATIONS}")

In [ ]:
# ============================================================
# 12. Save checkpoint
# ============================================================

CHECKPOINT_PATH = Path(f"/content/drive/MyDrive/stea_tkgl_{LANG_PAIR}_checkpoint.pt")

torch.save({
    "model_state_dict": model.state_dict(),
    "config": {
        "LANG_PAIR": LANG_PAIR,
        "TRAIN_SEED": TRAIN_SEED,
        "DIM": DIM,
        "DEPTH": DEPTH,
        "MARGIN": MARGIN,
        "DROPOUT": DROPOUT,
        "ALPHA": ALPHA,
        "LR": LR,
        "EPOCHS_PER_ITER": EPOCHS_PER_ITER,
        "ITERATIONS": ITERATIONS,
        "N1": N1,
        "N2": N2,
        "NUM_RELATIONS": NUM_RELATIONS,
    },
    "current_train_pairs": current_train_pairs,
    "final_metrics": final_result["left_to_right"],
}, CHECKPOINT_PATH)

print("Saved:", CHECKPOINT_PATH)

## Notes

- Set `LANG_PAIR` to `en_fa`, `en_zh`, `en_ja`, or `en_fr` and re-run from the config cell.
- Full TKGL graphs are large; use a **High-RAM GPU** runtime when possible.
- Debug faster with `MAX_QUADS_EN = 500_000` and `MAX_QUADS_OTHER = 200_000`.
- Intermediate eval uses up to 5k pairs; the final cell scores all held-out pairs.
- Mutual-NN only considers entities in `alignment_all.tsv`.
- Keep the same `TRAIN_SEED` across methods for fair comparison.